## Setup

In [3]:
from collections import Counter
import os
import numpy as np
import pandas as pd
import tensorflow as tf
from pathlib import Path
from csv import QUOTE_NONE

## Load

In [4]:
LOCAL_DATA_ROOT = Path(os.environ.get("MIND_DATA_ROOT", "./smallDataset")).expanduser().resolve()

train_dir = LOCAL_DATA_ROOT / f"MINDsmall_train"
valid_dir = LOCAL_DATA_ROOT / f"MINDsmall_dev"

train_news_path      = str(train_dir / "news.tsv")
train_behaviors_path = str(train_dir / "behaviors.tsv")
valid_news_path      = str(valid_dir / "news.tsv")
valid_behaviors_path = str(valid_dir / "behaviors.tsv")

NEWS_COLUMNS = [
    "nid",
    "vertical",
    "subvertical",
    "title",
    "abstract",
    "url",
    "title_entities",
    "abstract_entities",
]
BEHAVIOR_COLUMNS = [
    "impression_id",
    "user_id",
    "time",
    "history",
    "impressions",
]

def load_tsv(path_str: str, columns):
    return pd.read_table(
        path_str,
        header=None,
        names=columns,
        sep='\t',
        quoting=QUOTE_NONE,
        dtype=object,
        na_filter=False,
    )

train_news_df = load_tsv(train_news_path, NEWS_COLUMNS)
valid_news_df = load_tsv(valid_news_path, NEWS_COLUMNS)
train_behaviors_df = load_tsv(train_behaviors_path, BEHAVIOR_COLUMNS)
valid_behaviors_df = load_tsv(valid_behaviors_path, BEHAVIOR_COLUMNS)

print(f"Loaded {len(train_news_df)} train news rows and {len(valid_news_df)} validation news rows from {LOCAL_DATA_ROOT}")
print(f"Loaded {len(train_behaviors_df)} train behaviors rows and {len(valid_behaviors_df)} validation behaviors rows")


Loaded 51282 train news rows and 42416 validation news rows from /home/asmvas/Documents/NTNU-prosjekt/anbefalingssytem/AnbSysProsjekt/smallDataset
Loaded 156965 train behaviors rows and 73152 validation behaviors rows



## Most-popular baseline

Estimate article popularity from the training impressions and reuse that global
ranking as a simple non-personalized recommender. The validation impressions make
it easy to benchmark the baseline with ranking metrics such as nDCG, Hit@K, and MRR.


In [6]:
from typing import List, Tuple

def parse_impression_tokens(raw_impressions: str) -> List[Tuple[str, int]]:
    tokens: List[Tuple[str, int]] = []
    if not isinstance(raw_impressions, str):
        return tokens
    for token in raw_impressions.split():
        if "-" not in token:
            continue
        nid, label = token.rsplit("-", 1)
        if not nid:
            continue
        try:
            clicked = int(label)
        except ValueError:
            continue
        tokens.append((nid, 1 if clicked > 0 else 0))
    return tokens


def build_popularity_stats(behaviors_df: pd.DataFrame) -> pd.DataFrame:
    click_counter = Counter()
    display_counter = Counter()
    for impression_str in behaviors_df["impressions"]:
        for nid, clicked in parse_impression_tokens(impression_str):
            display_counter[nid] += 1
            if clicked:
                click_counter[nid] += 1
    stats = (
        pd.DataFrame(
            {
                "nid": list(display_counter.keys()),
                "impressions": list(display_counter.values()),
            }
        )
        .assign(clicks=lambda df: df["nid"].map(lambda nid: click_counter.get(nid, 0)))
    )
    stats["ctr"] = stats["clicks"] / stats["impressions"].replace(0, np.nan)
    stats["ctr"] = stats["ctr"].fillna(0.0)
    stats["score"] = stats["clicks"] + stats["ctr"]
    stats = stats.sort_values(by=["score", "impressions"], ascending=False).reset_index(drop=True)
    return stats


popularity_stats = build_popularity_stats(train_behaviors_df)
popular_rank = {nid: rank for rank, nid in enumerate(popularity_stats["nid"].tolist())}
fallback_rank = len(popular_rank)


def _dcg_at_k(labels: np.ndarray, k: int) -> float:
    if k <= 0 or labels.size == 0:
        return 0.0
    k = min(k, labels.size)
    discounts = 1.0 / np.log2(np.arange(2, k + 2))
    return float(np.sum(labels[:k] * discounts))


def _pairwise_auc(labels: np.ndarray, scores: np.ndarray):
    pos_scores = scores[labels == 1]
    neg_scores = scores[labels == 0]
    if pos_scores.size == 0 or neg_scores.size == 0:
        return None
    comparisons = pos_scores[:, None] - neg_scores[None, :]
    wins = (comparisons > 0).sum()
    ties = (comparisons == 0).sum()
    total_pairs = pos_scores.size * neg_scores.size
    return float((wins + 0.5 * ties) / total_pairs)


def evaluate_most_popular(
    behaviors_df: pd.DataFrame,
    k_values: Tuple[int, ...] = (5, 10),
) -> dict:
    ndcgs = {k: [] for k in k_values}
    hits = {k: [] for k in k_values}
    mrrs, aucs = [], []
    impressions_used = 0
    for impression_str in behaviors_df["impressions"]:
        parsed = parse_impression_tokens(impression_str)
        if not parsed:
            continue
        candidate_ids = [nid for nid, _ in parsed]
        labels = np.array([label for _, label in parsed], dtype=np.float32)
        ranks = np.array([popular_rank.get(nid, fallback_rank) for nid in candidate_ids], dtype=np.int32)
        scores = -ranks.astype(np.float32)
        order = np.argsort(ranks, kind="mergesort")
        sorted_labels = labels[order]
        ideal_labels = np.sort(labels)[::-1]
        for k in k_values:
            ideal_dcg = _dcg_at_k(ideal_labels, k)
            ndcg = _dcg_at_k(sorted_labels, k) / ideal_dcg if ideal_dcg > 0 else 0.0
            ndcgs[k].append(ndcg)
            hits[k].append(1.0 if sorted_labels[:k].sum() > 0 else 0.0)
        positives = np.where(sorted_labels == 1)[0]
        mrrs.append(1.0 / (positives[0] + 1) if positives.size > 0 else 0.0)
        auc = _pairwise_auc(labels, scores)
        if auc is not None:
            aucs.append(auc)
        impressions_used += 1
    metrics = {f"ndcg@{k}": float(np.mean(values)) if values else 0.0 for k, values in ndcgs.items()}
    metrics.update({f"hit@{k}": float(np.mean(values)) if values else 0.0 for k, values in hits.items()})
    metrics["mrr"] = float(np.mean(mrrs)) if mrrs else 0.0
    metrics["auc"] = float(np.mean(aucs)) if aucs else 0.0
    metrics["impressions_evaluated"] = impressions_used
    return metrics


top_popular = popularity_stats.loc[:9, ["nid", "clicks", "impressions", "ctr"]].copy()
top_popular["ctr"] = top_popular["ctr"].round(4)
print("Top globally popular news items (training split):")
print(top_popular.to_string(index=False))

baseline_metrics = evaluate_most_popular(valid_behaviors_df)
baseline_metrics


Top globally popular news items (training split):
   nid  clicks  impressions    ctr
N55689    4316        18315 0.2357
N35729    3346        15418 0.2170
N33619    3246        15062 0.2155
N53585    2835         9908 0.2861
N63970    2578        14276 0.1806
N49685    2294         7229 0.3173
N49279    2270         6229 0.3644
  N287    2128        10019 0.2124
N23446    1930        15500 0.1245
N51048    1875        19242 0.0974


{'ndcg@5': 0.24517091822982462,
 'ndcg@10': 0.30765610911837166,
 'hit@5': 0.4196194225721785,
 'hit@10': 0.6115895669291339,
 'mrr': 0.26518879846752524,
 'auc': 0.5240215149779247,
 'impressions_evaluated': 73152}